In [1]:
import json
import random
import re
import ast
import random
from collections import defaultdict
from pathlib import Path
from transformers import AutoTokenizer

ABS_REL_DIR = Path("/home/lucas/Desktop/UCSD/Research/sequential-decision-processors/verl_dead_agent/agent_system/environments")
tok_path = ABS_REL_DIR / "tokenizers" / "qwen3"
tok = AutoTokenizer.from_pretrained(tok_path, use_fast=True, local_files_only=True)

/home/lucas/Desktop/UCSD/Research/sequential-decision-processors/verl_dead_agent/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def load_jsonl(path: Path):
    data = []
    jsonl_files = sorted(path.glob("*.jsonl"), key=lambda x: int(x.stem) if x.stem.isdigit() else float('inf'))    
    for jsonl_file in jsonl_files:
        with jsonl_file.open("r", encoding="utf-8") as f:
            for line in f:
                if not line:
                    continue
                sample = json.loads(line)
                if sample.get('score', 0) >= 1:
                    data.append(sample)
    
    print(f"Loaded {len(data)} samples from {len(jsonl_files)} files.")
    return data

# load our data in
data = load_jsonl(Path("rejection_sampling/twx_8"))

Loaded 1178 samples from 1 files.


In [3]:
# # Optionally can print out a sample
# for k,v in data[0].items():
#     print(f"{k}\n{v}\n\n")

In [4]:
# Now converting data to my cleaned sft format
def process_and_write(data, filename, sys_prompt_name):
    cleaned_path = Path("cleaned_sft")
    cleaned_path.mkdir(exist_ok=True)
    out_file = cleaned_path / f"{filename}.jsonl"
    with out_file.open("w", encoding="utf-8") as f:
        for d in data:
            inp = d.get("input", "").strip()
            # Remove the starting 'user\n' and ending '\nassistant'
            inp = inp.removeprefix("user\n").removesuffix("\nassistant").strip()
            out = d.get("output", "").strip()

            chat = [
                ["system", sys_prompt_name],
                ["user", inp],
                ["assistant", out]
            ]

            json.dump({"chat": chat}, f, ensure_ascii=False)
            f.write("\n")

In [5]:
# process_and_write(data, f"rft_twx_{len(data)}", sys_prompt_name="tw_general.txt")

# Clean our Gold Path Data

In [6]:
def load_jsonl(path: Path):
    data = []
    jsonl_files = sorted(path.glob("*.jsonl"), key=lambda x: int(x.stem) if x.stem.isdigit() else float('inf'))    
    for jsonl_file in jsonl_files:
        with jsonl_file.open("r", encoding="utf-8") as f:
            for line in f:
                sample = json.loads(line)
                data.append(sample)
    
    print(f"Loaded {len(data)} samples from {len(jsonl_files)} files.")
    return data

# load our data in
data = load_jsonl(Path("gold_path/"))

Loaded 30387 samples from 2 files.


### Best Move

In [ ]:
SAVE_BESTMOVE_DATA = False
best_move_data = []
sys_prompt = "tw_bestmove.txt"
cleaned_path = Path("cleaned_sft")
filename = f"bestmove_{len(data)}.jsonl"
out_path = cleaned_path / filename

if SAVE_BESTMOVE_DATA:
    with out_path.open("w", encoding="utf-8") as f:
        for d in data:
            chat = [
                ["system", sys_prompt],
                ["user", d["obs"]],
                ["assistant", d["info"]["step_info"]["action"]],
            ]
            json.dump({"chat": chat}, f, ensure_ascii=False)
            f.write("\n")

### Best Line

In [11]:
# Start by grouping by trajectories
def group_and_sort(data):
    groups = defaultdict(list)
    for d in data:
        run_info = d["info"]["run_info"]
        key = (run_info["seed"], run_info["proc_id"])
        groups[key].append(d)
    for key in groups:
        groups[key] = sorted(groups[key], key=lambda x: x["info"]["run_info"]["step"])
    group_list = list(groups.values())
    print(f"Total groups: {len(group_list) }")
    return group_list

# Get sorted / grouped by seed + proc_id
sorted_groups = group_and_sort(data)

Total groups: 2200


In [ ]:
SAVE_BESTLINE_DATA = True
best_line_data = []
sys_prompt = "tw_bestline.txt"
cleaned_path = Path("cleaned_sft")
filename = f"bestline_{len(data)}.jsonl"
out_path = cleaned_path / filename

if SAVE_BESTLINE_DATA:
    with out_path.open("w", encoding="utf-8") as f:
        for d in sorted_groups:
            chat = [
                ["system", sys_prompt],
                ["user", d["obs"]],
                ["assistant", d["info"]["step_info"]["action"]],
            ]
            json.dump({"chat": chat}, f, ensure_ascii=False)
            f.write("\n")

{'obs': "You are an agent operating in AlfWorld, an interactive-fiction, text-world environment.\nYou should first reason about your current situation prior to returning your chosen action. You MUST format your thinking as <think> your_reasoning </think> and your action as <action> your_action </action>. \nIf you do not enclose your reasoning and action within their respective tags, your response will be rejected. You can only provide one action at a time.\nFor example, <think> my_thinking... </think> <action> take lantern </action>.\nNote that you must move to an object before you can interact with it. You can use 'inventory' to get your current inventory. You should reference each object by its precise name -- e.g., 'mug 3'.\nThe set of action templates are the following: ['clean _ with _', 'close _', 'cool _ with _', 'examine _', 'go to _', 'heat _ with _', 'move _ to _', 'open _', 'slice _ with _', 'take _ from _', 'use _', 'inventory', 'get legal moves']. You should only use these